# Tesseract Maltese (mlt) OCR Test
This notebook loads OCR test records from `mlt_data/ocr_data_v11_test.json`, reads cropped box images from `mlt_data/crops/test`, and runs Tesseract OCR with `lang="mlt"`.

## 1. Environment And Imports
This step imports all required libraries for file handling, OCR, and tabular analysis.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import urllib.request

import pandas as pd
import pytesseract
from PIL import Image

## 2. Dataset And OCR Configuration
Set workspace paths, select the JSON test split, point to image files, and set OCR language to `mlt`.

In [ ]:
JSON_PATH = "path/to/your/json/file"  # Update this path to your actual dev split JSON file
CROPS_DIR = "path/to/your/crops/directory"  # Update this path to your actual crops directory
LANG = "mlt"
TESSDATA_DIR = "path/to/your/tessdata/directory"  # Update this path to your actual tessdata directory

print(f"JSON exists: {JSON_PATH.exists()} -> {JSON_PATH}")
print(f"Crops dir exists: {CROPS_DIR.exists()} -> {CROPS_DIR}")

## 3. Verify Tesseract Language Data
Check whether the `mlt` traineddata is installed; if missing, download and configure it automatically.

In [ ]:
lang_output = subprocess.run(["tesseract", "--list-langs"], capture_output=True, text=True, check=True).stdout
available_langs = {line.strip() for line in lang_output.splitlines() if line.strip() and not line.startswith("List of available languages")}

if LANG not in available_langs:
    TESSDATA_DIR.mkdir(exist_ok=True)
    traineddata_path = TESSDATA_DIR / f"{LANG}.traineddata"
    if not traineddata_path.exists():
        url = f"https://github.com/tesseract-ocr/tessdata_best/raw/main/{LANG}.traineddata"
        urllib.request.urlretrieve(url, traineddata_path)
    os.environ["TESSDATA_PREFIX"] = str(TESSDATA_DIR)
    print(f"Downloaded and configured {LANG}.traineddata in {TESSDATA_DIR}")
else:
    print(f"Language '{LANG}' is already available in this environment.")

print("TESSDATA_PREFIX:", os.environ.get("TESSDATA_PREFIX", "<system default>"))

## 4. Load Test Records And Resolve Crop Paths
Parse test records from JSON, map each box to a crop file path, and validate that each crop exists in the crops test directory.

In [ ]:
with JSON_PATH.open("r", encoding="utf-8") as f:
    test_data = json.load(f)

if isinstance(test_data, dict):
    records = test_data.get("data", [])
elif isinstance(test_data, list):
    records = test_data
else:
    records = []

rows = []
for item in records:
    if not isinstance(item, dict):
        continue

    page = item.get("page", {})
    fname = page.get("page_fname")
    if not fname:
        continue

    base_id = Path(fname).stem
    doc_id = str(page.get("document_id") or base_id.split("-")[0])
    page_id = str(page.get("page_id") or base_id.split("-")[1])
    boxes = item.get("boxes", [])

    for box_idx, box_item in enumerate(boxes):
        crop_path = CROPS_DIR / doc_id / page_id / f"{base_id}_box{box_idx:04d}.jpg"
        gt_text = box_item.get("transcription", "") if isinstance(box_item, dict) else ""
        rows.append({
            "page_fname": fname,
            "box_idx": box_idx,
            "crop_path": str(crop_path),
            "exists": crop_path.exists(),
            "gt_text": gt_text,
        })

df_crops = pd.DataFrame(rows).sort_values(["page_fname", "box_idx"]).reset_index(drop=True)
print(f"Crop records parsed: {len(df_crops)}")
print(f"Missing crop images: {(~df_crops['exists']).sum()}")
df_crops.head(12)

## 5. Run OCR On Selected Crops
Run Tesseract on cropped box images, then inspect OCR predictions against ground-truth snippets.

In [ ]:
MAX_CROPS = None  # Set to None to run all crops

to_run = df_crops[df_crops["exists"]].copy()
if MAX_CROPS is not None:
    to_run = to_run.head(MAX_CROPS)

results = []
for _, row in to_run.iterrows():
    crop_path = Path(row["crop_path"])
    ocr_text = pytesseract.image_to_string(Image.open(crop_path), lang=LANG).strip()
    results.append({
        "page_fname": row["page_fname"],
        "box_idx": row["box_idx"],
        "crop_path": str(crop_path),
        "gt_text": row["gt_text"],
        "pred_text": ocr_text,
        "num_chars": len(ocr_text),
        "gt_preview": str(row["gt_text"]).replace("\n", " ")[:180],
        "ocr_preview": ocr_text.replace("\n", " ")[:180],
    })

df_ocr = pd.DataFrame(results)
print(f"Evaluated crops: {len(df_ocr)}")
df_ocr[["page_fname", "box_idx", "gt_preview", "ocr_preview"]].head(12)

## 6. Evaluate OCR Quality With CER
Compute crop-level CER by comparing each OCR prediction to its corresponding ground-truth transcription.

In [ ]:
# CER evaluation at crop level
import unicodedata

def normalize_text(s: str) -> str:
    s = unicodedata.normalize("NFC", str(s))
    s = " ".join(s.split())
    return s

def levenshtein_distance(a: str, b: str) -> int:
    if a == b:
        return 0
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            ins = curr[j - 1] + 1
            dele = prev[j] + 1
            sub = prev[j - 1] + (ca != cb)
            curr.append(min(ins, dele, sub))
        prev = curr
    return prev[-1]

df_eval = df_ocr.copy()
df_eval["gt_text"] = df_eval["gt_text"].fillna("")
df_eval["pred_text"] = df_eval["pred_text"].fillna("")
df_eval["gt_norm"] = df_eval["gt_text"].map(normalize_text)
df_eval["pred_norm"] = df_eval["pred_text"].map(normalize_text)

df_eval["gt_len"] = df_eval["gt_norm"].str.len()
df_eval["edit_distance"] = [
    levenshtein_distance(pred, gt)
    for pred, gt in zip(df_eval["pred_norm"], df_eval["gt_norm"])
]
df_eval["cer"] = df_eval["edit_distance"] / df_eval["gt_len"].replace(0, pd.NA)

weighted_cer = df_eval["edit_distance"].sum() / max(df_eval["gt_len"].sum(), 1)
mean_cer = df_eval["cer"].dropna().mean()

print(f"Evaluated crops: {len(df_eval)}")
print(f"Weighted CER: {weighted_cer:.4f}")
print(f"Mean CER (per-crop): {mean_cer:.4f}")

df_eval[["page_fname", "box_idx", "gt_len", "edit_distance", "cer", "gt_text", "pred_text"]].sort_values("cer", ascending=False).reset_index(drop=True)

## 7. Analyzing Page Outputs

In [ ]:
# Visualize crops with GT and prediction for a specific page, plus page-level weighted CER
from IPython.display import display, Markdown

PAGE_ID = "0100-002"  # e.g. 0003-003 or 0003-003.jpg
MAX_SHOW = None  # Set to None to show all crops for that page

target_page = PAGE_ID if PAGE_ID.endswith(".jpg") else f"{PAGE_ID}.jpg"
page_rows = df_eval[df_eval["page_fname"] == target_page].sort_values("box_idx").reset_index(drop=True)

if MAX_SHOW is not None:
    page_rows = page_rows.head(MAX_SHOW)

print(f"Page: {target_page}")
print(f"Crops shown: {len(page_rows)}")

if page_rows.empty:
    print("No rows found. Check PAGE_ID and ensure OCR/CER cells were run first.")
else:
    total_edits = page_rows["edit_distance"].sum()
    total_gt_len = int(page_rows["gt_len"].sum())
    page_weighted_cer = total_edits / max(total_gt_len, 1)
    page_mean_cer = page_rows["cer"].dropna().mean()

    print(f"Weighted CER for page: {page_weighted_cer:.4f}")
    print(f"Mean CER for page (per-crop): {page_mean_cer:.4f}")
    print("=" * 80)

    # for _, row in page_rows.iterrows():
    #     crop_path = Path(row["crop_path"])

    #     display(Markdown(f"### box {int(row['box_idx']):04d} | CER: {float(row['cer']):.4f}"))
    #     display(Image.open(crop_path))
    #     display(Markdown("**GT**"))
    #     print(str(row["gt_text"]))
    #     display(Markdown("**Prediction**"))
    #     print(str(row["pred_text"]))
    #     print("-" * 80)

In [ ]:
# Excel-friendly TSV output: one row per BB with GT, Prediction, and CER columns

def join_lines(lines: list[str]) -> str:
    if lines == []:
        return ""

    lines = [
        line
        for line in (
            line.strip()
            for line in lines
        )
        if line != ""
    ]

    if lines == []:
        return ""

    new_lines = []
    last_line_index = len(lines) - 1
    for i, line in enumerate(lines):
        new_lines.append(line)
        if (
            i != last_line_index and (
                line[-1] not in "-—"
                or (len(line) > 1 and line[-2] == " ")
            )
        ):
            new_lines.append(" ")

    return "".join(new_lines)


def _clean_text(v):
    if v is None:
        return ""
    return join_lines(str(v).splitlines())

# Prefer the currently selected page; fall back to all evaluated rows.
if "page_rows" in globals() and page_rows is not None and len(page_rows) > 0:
    src = page_rows.copy()
else:
    src = df_eval.copy()

needed = [c for c in ["page_fname", "box_idx", "gt_text", "pred_text", "cer"] if c in src.columns]
out = src[needed].copy()

if "page_fname" not in out.columns:
    out["page_fname"] = ""
if "box_idx" not in out.columns:
    out["box_idx"] = range(len(out))
if "cer" not in out.columns:
    out["cer"] = pd.NA

out = out.sort_values(["page_fname", "box_idx"], kind="stable")

export_df = out.assign(
    value=out.apply(lambda r: f"page:{_clean_text(r.get('page_fname', ''))} bb:{int(r.get('box_idx', 0)):04d}", axis=1),
    GT=out["gt_text"].map(_clean_text),
    Prediction=out["pred_text"].map(_clean_text),
    CER=out["cer"].map(lambda x: "" if pd.isna(x) else f"{float(x):.4f}"),
)[["value", "GT", "Prediction", "CER"]]

# Print as tab-separated values so copy/paste lands in Excel columns.
print(export_df.to_csv(sep="\t", index=False))

## 8. Worst-Page CER Analysis
Automatically find the page with the highest weighted CER and analyze it using the same page-level view as Section 7.

In [ ]:
from IPython.display import display, Markdown

page_summary = (
    df_eval.groupby("page_fname", dropna=False)
    .agg(
        total_edits=("edit_distance", "sum"),
        total_gt_len=("gt_len", "sum"),
        mean_cer=("cer", "mean"),
        num_boxes=("box_idx", "count"),
    )
    .reset_index()
)
page_summary["weighted_cer"] = page_summary["total_edits"] / page_summary["total_gt_len"].clip(lower=1)
page_summary = page_summary.sort_values(["weighted_cer", "mean_cer", "page_fname"], ascending=[False, False, True]).reset_index(drop=True)

worst_page = str(page_summary.loc[0, "page_fname"])
worst_page_rows = df_eval[df_eval["page_fname"] == worst_page].sort_values("box_idx").reset_index(drop=True)
worst_page_weighted_cer = float(page_summary.loc[0, "weighted_cer"])
worst_page_mean_cer = float(page_summary.loc[0, "mean_cer"])
worst_page_boxes = int(page_summary.loc[0, "num_boxes"])

print(f"Worst page by weighted CER: {worst_page}")
print(f"Crops shown: {worst_page_boxes}")
print(f"Weighted CER for page: {worst_page_weighted_cer:.4f}")
print(f"Mean CER for page (per-crop): {worst_page_mean_cer:.4f}")
print("=" * 80)

for _, row in worst_page_rows.iterrows():
    crop_path = Path(row["crop_path"])

    # display(Markdown(f"### box {int(row['box_idx']):04d} | CER: {float(row['cer']):.4f}"))
    # display(Image.open(crop_path))
    # display(Markdown("**GT**"))
    # print(str(row["gt_text"]))
    # display(Markdown("**Prediction**"))
    # print(str(row["pred_text"]))
    # print("-" * 80)

In [ ]:
PAGE_ID = "0060-014"  # e.g. 0003-003 or 0003-003.jpg
MAX_SHOW = None  # Set to None to show all crops for that page

target_page = PAGE_ID if PAGE_ID.endswith(".jpg") else f"{PAGE_ID}.jpg"
page_rows = df_eval[df_eval["page_fname"] == target_page].sort_values("box_idx").reset_index(drop=True)


In [ ]:
# Excel-friendly TSV output: one row per BB with GT, Prediction, and CER columns

def join_lines(lines: list[str]) -> str:
    if lines == []:
        return ""

    lines = [
        line
        for line in (
            line.strip()
            for line in lines
        )
        if line != ""
    ]

    if lines == []:
        return ""

    new_lines = []
    last_line_index = len(lines) - 1
    for i, line in enumerate(lines):
        new_lines.append(line)
        if (
            i != last_line_index and (
                line[-1] not in "-—"
                or (len(line) > 1 and line[-2] == " ")
            )
        ):
            new_lines.append(" ")

    return "".join(new_lines)


def _clean_text(v):
    if v is None:
        return ""
    return join_lines(str(v).splitlines())

# Prefer the currently selected page; fall back to all evaluated rows.
if "page_rows" in globals() and page_rows is not None and len(page_rows) > 0:
    src = page_rows.copy()
else:
    src = df_eval.copy()

needed = [c for c in ["page_fname", "box_idx", "gt_text", "pred_text", "cer"] if c in src.columns]
out = src[needed].copy()

if "page_fname" not in out.columns:
    out["page_fname"] = ""
if "box_idx" not in out.columns:
    out["box_idx"] = range(len(out))
if "cer" not in out.columns:
    out["cer"] = pd.NA

out = out.sort_values(["page_fname", "box_idx"], kind="stable")

export_df = out.assign(
    value=out.apply(lambda r: f"page:{_clean_text(r.get('page_fname', ''))} bb:{int(r.get('box_idx', 0)):04d}", axis=1),
    GT=out["gt_text"].map(_clean_text),
    Prediction=out["pred_text"].map(_clean_text),
    CER=out["cer"].map(lambda x: "" if pd.isna(x) else f"{float(x):.4f}"),
)[["value", "GT", "Prediction", "CER"]]

# Print as tab-separated values so copy/paste lands in Excel columns.
print(export_df.to_csv(sep="\t", index=False))